# 🧊 Apache Iceberg com Apache Spark

Este notebook demonstra o uso do **Apache Iceberg** integrado ao **Apache Spark (PySpark)** com um cenário de **E-commerce**.

## 📦 Cenário: E-commerce

Tabelas utilizadas:
- `clientes` — cadastro de clientes
- `produtos` — catálogo de produtos
- `pedidos` — pedidos realizados

## 🗺️ Modelo ER

```
+------------+       +------------+       +------------+
|  clientes  |       |  pedidos   |       |  produtos  |
+------------+       +------------+       +------------+
| id (PK)    |<------| cliente_id |       | id (PK)    |
| nome       |       | id (PK)    |------>| nome       |
| email      |       | produto_id |       | categoria  |
| cidade     |       | quantidade |       | preco      |
| ativo      |       | status     |       | estoque    |
+------------+       | total      |       +------------+
                     | data       |
                     +------------+
```

## 📋 DDL das Tabelas

```sql
-- Tabela clientes
CREATE TABLE local.ecommerce.clientes (
    id     INT,
    nome   STRING,
    email  STRING,
    cidade STRING,
    ativo  BOOLEAN
) USING iceberg;

-- Tabela produtos
CREATE TABLE local.ecommerce.produtos (
    id        INT,
    nome      STRING,
    categoria STRING,
    preco     DOUBLE,
    estoque   INT
) USING iceberg;

-- Tabela pedidos
CREATE TABLE local.ecommerce.pedidos (
    id         INT,
    cliente_id INT,
    produto_id INT,
    quantidade INT,
    status     STRING,
    total      DOUBLE,
    data       STRING
) USING iceberg;
```

## 1️⃣ Configuração da Sessão Spark com Apache Iceberg

In [2]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession

ICEBERG_VERSION = "1.5.2"
SCALA_VERSION   = "2.12"
SPARK_VERSION   = "3.5"

spark = (
    SparkSession.builder
    .appName("Apache Iceberg - E-commerce")
    .master("local[*]")
    # Pacote Iceberg
    .config(
        "spark.jars.packages",
        f"org.apache.iceberg:iceberg-spark-runtime-{SPARK_VERSION}_{SCALA_VERSION}:{ICEBERG_VERSION}"
    )
    # Extensões SQL
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catálogo local (HadoopCatalog)
    .config("spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "./data/iceberg/warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark versão: {spark.version}")
print(f"✅ Sessão Iceberg iniciada com sucesso!")

✅ Spark versão: 3.5.1
✅ Sessão Iceberg iniciada com sucesso!


## 2️⃣ Criando o Namespace (Database)

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ecommerce")
spark.sql("SHOW NAMESPACES IN local").show()
print("✅ Namespace 'local.ecommerce' criado!")

+---------+
|namespace|
+---------+
|ecommerce|
+---------+

✅ Namespace 'local.ecommerce' criado!


## 3️⃣ INSERT — Criando e Populando as Tabelas Iceberg

In [4]:
# ── Tabela CLIENTES ──────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS local.ecommerce.clientes")
spark.sql("""
    CREATE TABLE local.ecommerce.clientes (
        id     INT,
        nome   STRING,
        email  STRING,
        cidade STRING,
        ativo  BOOLEAN
    ) USING iceberg
""")

spark.sql("""
    INSERT INTO local.ecommerce.clientes VALUES
        (1, 'Ana Silva',     'ana@email.com',    'São Paulo',      true),
        (2, 'Bruno Costa',   'bruno@email.com',  'Rio de Janeiro', true),
        (3, 'Carla Souza',   'carla@email.com',  'Curitiba',       true),
        (4, 'Diego Lima',    'diego@email.com',  'Florianópolis',  true),
        (5, 'Elena Martins', 'elena@email.com',  'Porto Alegre',   true)
""")

print("✅ Tabela 'clientes' criada e populada!")
spark.sql("SELECT * FROM local.ecommerce.clientes").show()

✅ Tabela 'clientes' criada e populada!
+---+-------------+---------------+--------------+-----+
| id|         nome|          email|        cidade|ativo|
+---+-------------+---------------+--------------+-----+
|  1|    Ana Silva|  ana@email.com|     São Paulo| true|
|  2|  Bruno Costa|bruno@email.com|Rio de Janeiro| true|
|  3|  Carla Souza|carla@email.com|      Curitiba| true|
|  4|   Diego Lima|diego@email.com| Florianópolis| true|
|  5|Elena Martins|elena@email.com|  Porto Alegre| true|
+---+-------------+---------------+--------------+-----+



In [6]:
spark.sql("DROP TABLE IF EXISTS local.ecommerce.produtos")
spark.sql("""
    CREATE TABLE local.ecommerce.produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     DOUBLE,
        estoque   INT
    ) USING iceberg
""")

spark.sql("""
    INSERT INTO local.ecommerce.produtos VALUES
        (1, 'Notebook Dell',    'Eletronicos', 3500.00, 15),
        (2, 'Mouse Logitech',   'Perifericos',   89.90, 100),
        (3, 'Teclado Mecanico', 'Perifericos',  250.00,  50),
        (4, 'Monitor 24pol',    'Eletronicos', 1200.00,  20),
        (5, 'Headset Gamer',    'Perifericos',  350.00,  35)
""")

print("✅ Tabela 'produtos' criada e populada!")
spark.sql("SELECT * FROM local.ecommerce.produtos").show()

✅ Tabela 'produtos' criada e populada!
+---+----------------+-----------+------+-------+
| id|            nome|  categoria| preco|estoque|
+---+----------------+-----------+------+-------+
|  1|   Notebook Dell|Eletronicos|3500.0|     15|
|  2|  Mouse Logitech|Perifericos|  89.9|    100|
|  3|Teclado Mecanico|Perifericos| 250.0|     50|
|  4|   Monitor 24pol|Eletronicos|1200.0|     20|
|  5|   Headset Gamer|Perifericos| 350.0|     35|
+---+----------------+-----------+------+-------+



In [7]:
# ── Tabela PEDIDOS ───────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS local.ecommerce.pedidos")
spark.sql("""
    CREATE TABLE local.ecommerce.pedidos (
        id         INT,
        cliente_id INT,
        produto_id INT,
        quantidade INT,
        status     STRING,
        total      DOUBLE,
        data       STRING
    ) USING iceberg
""")

spark.sql("""
    INSERT INTO local.ecommerce.pedidos VALUES
        (1, 1, 1, 1, 'pendente',  3500.00, '2024-01-10'),
        (2, 2, 2, 3, 'aprovado',   269.70, '2024-01-11'),
        (3, 3, 3, 1, 'pendente',   250.00, '2024-01-12'),
        (4, 4, 4, 2, 'entregue',  2400.00, '2024-01-13'),
        (5, 5, 5, 1, 'pendente',   350.00, '2024-01-14')
""")

print("✅ Tabela 'pedidos' criada e populada!")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Tabela 'pedidos' criada e populada!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
+---+----------+----------+----------+--------+------+----------+



In [8]:
# ── INSERT de novos pedidos ──────────────────────────────────
spark.sql("""
    INSERT INTO local.ecommerce.pedidos VALUES
        (6, 1, 2, 2, 'aprovado', 179.80, '2024-01-15'),
        (7, 3, 5, 1, 'pendente', 350.00, '2024-01-15')
""")

print("✅ Novos pedidos inseridos!")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Novos pedidos inseridos!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|pendente| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



## 4️⃣ UPDATE — Atualizando Registros

In [9]:
# ── UPDATE: Aprovar pedidos pendentes ────────────────────────
spark.sql("""
    UPDATE local.ecommerce.pedidos
    SET status = 'aprovado'
    WHERE status = 'pendente'
""")

print("✅ Pedidos 'pendente' atualizados para 'aprovado'!")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Pedidos 'pendente' atualizados para 'aprovado'!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



In [10]:
# ── UPDATE: Reajuste de preço nos Eletrônicos (+10%) ─────────
spark.sql("""
    UPDATE local.ecommerce.produtos
    SET preco = preco * 1.10
    WHERE categoria = 'Eletrônicos'
""")

print("✅ Preços de Eletrônicos reajustados em +10%!")
spark.sql("SELECT * FROM local.ecommerce.produtos").show()

✅ Preços de Eletrônicos reajustados em +10%!
+---+----------------+-----------+------+-------+
| id|            nome|  categoria| preco|estoque|
+---+----------------+-----------+------+-------+
|  1|   Notebook Dell|Eletronicos|3500.0|     15|
|  2|  Mouse Logitech|Perifericos|  89.9|    100|
|  3|Teclado Mecanico|Perifericos| 250.0|     50|
|  4|   Monitor 24pol|Eletronicos|1200.0|     20|
|  5|   Headset Gamer|Perifericos| 350.0|     35|
+---+----------------+-----------+------+-------+



In [11]:
# ── UPDATE: Desativar cliente ────────────────────────────────
spark.sql("""
    UPDATE local.ecommerce.clientes
    SET ativo = false
    WHERE email = 'diego@email.com'
""")

print("✅ Cliente 'Diego Lima' desativado!")
spark.sql("SELECT * FROM local.ecommerce.clientes").show()

✅ Cliente 'Diego Lima' desativado!
+---+-------------+---------------+--------------+-----+
| id|         nome|          email|        cidade|ativo|
+---+-------------+---------------+--------------+-----+
|  1|    Ana Silva|  ana@email.com|     São Paulo| true|
|  4|   Diego Lima|diego@email.com| Florianópolis|false|
|  2|  Bruno Costa|bruno@email.com|Rio de Janeiro| true|
|  3|  Carla Souza|carla@email.com|      Curitiba| true|
|  5|Elena Martins|elena@email.com|  Porto Alegre| true|
+---+-------------+---------------+--------------+-----+



## 5️⃣ DELETE — Removendo Registros

In [12]:
# ── DELETE: Remover pedidos entregues ────────────────────────
spark.sql("""
    DELETE FROM local.ecommerce.pedidos
    WHERE status = 'entregue'
""")

print("✅ Pedidos 'entregue' removidos!")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Pedidos 'entregue' removidos!
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



In [13]:
# ── DELETE: Remover produto sem estoque ──────────────────────
spark.sql("""
    UPDATE local.ecommerce.produtos
    SET estoque = 0
    WHERE id = 3
""")

spark.sql("""
    DELETE FROM local.ecommerce.produtos
    WHERE estoque = 0
""")

print("✅ Produtos sem estoque removidos!")
spark.sql("SELECT * FROM local.ecommerce.produtos").show()

✅ Produtos sem estoque removidos!
+---+--------------+-----------+------+-------+
| id|          nome|  categoria| preco|estoque|
+---+--------------+-----------+------+-------+
|  1| Notebook Dell|Eletronicos|3500.0|     15|
|  2|Mouse Logitech|Perifericos|  89.9|    100|
|  4| Monitor 24pol|Eletronicos|1200.0|     20|
|  5| Headset Gamer|Perifericos| 350.0|     35|
+---+--------------+-----------+------+-------+



## 6️⃣ Snapshots — Viagem no Tempo (Time Travel)

In [14]:
# ── Listar snapshots da tabela pedidos ───────────────────────
print("📜 Snapshots da tabela 'pedidos':")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.ecommerce.pedidos.snapshots
""").show(truncate=False)

📜 Snapshots da tabela 'pedidos':
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|3219211101809182595|2026-05-03 04:29:42.742|append   |
|5585881754422269752|2026-05-03 04:29:44.459|append   |
|5437845665960675529|2026-05-03 04:29:47.392|overwrite|
|740927721801071722 |2026-05-03 04:29:52.177|delete   |
+-------------------+-----------------------+---------+



In [15]:
# ── Consulta por versão (snapshot_id) ────────────────────────
# Pegue o snapshot_id do primeiro snapshot da célula acima e substitua abaixo
snapshots = spark.sql("""
    SELECT snapshot_id FROM local.ecommerce.pedidos.snapshots ORDER BY committed_at
""").collect()

primeiro_snapshot = snapshots[0]["snapshot_id"]
print(f"🕐 Lendo snapshot inicial (id={primeiro_snapshot}):")

spark.read \
    .option("snapshot-id", primeiro_snapshot) \
    .table("local.ecommerce.pedidos") \
    .show()

🕐 Lendo snapshot inicial (id=3219211101809182595):
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|pendente|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|pendente| 250.0|2024-01-12|
|  4|         4|         4|         2|entregue|2400.0|2024-01-13|
|  5|         5|         5|         1|pendente| 350.0|2024-01-14|
+---+----------+----------+----------+--------+------+----------+



In [16]:
# ── Estado atual da tabela ───────────────────────────────────
print("✅ Estado atual da tabela 'pedidos':")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Estado atual da tabela 'pedidos':
+---+----------+----------+----------+--------+------+----------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|
+---+----------+----------+----------+--------+------+----------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|
+---+----------+----------+----------+--------+------+----------+



## 7️⃣ Schema Evolution — Adicionando uma Nova Coluna

In [17]:
# ── ALTER TABLE: adiciona coluna desconto ────────────────────
spark.sql("""
    ALTER TABLE local.ecommerce.pedidos
    ADD COLUMN desconto DOUBLE
""")

spark.sql("""
    UPDATE local.ecommerce.pedidos
    SET desconto = 0.0
""")

print("✅ Coluna 'desconto' adicionada via Schema Evolution!")
spark.sql("SELECT * FROM local.ecommerce.pedidos ORDER BY id").show()

✅ Coluna 'desconto' adicionada via Schema Evolution!
+---+----------+----------+----------+--------+------+----------+--------+
| id|cliente_id|produto_id|quantidade|  status| total|      data|desconto|
+---+----------+----------+----------+--------+------+----------+--------+
|  1|         1|         1|         1|aprovado|3500.0|2024-01-10|     0.0|
|  2|         2|         2|         3|aprovado| 269.7|2024-01-11|     0.0|
|  3|         3|         3|         1|aprovado| 250.0|2024-01-12|     0.0|
|  5|         5|         5|         1|aprovado| 350.0|2024-01-14|     0.0|
|  6|         1|         2|         2|aprovado| 179.8|2024-01-15|     0.0|
|  7|         3|         5|         1|aprovado| 350.0|2024-01-15|     0.0|
+---+----------+----------+----------+--------+------+----------+--------+



## 8️⃣ MERGE INTO — Upsert (Insert + Update)

In [18]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

# Dados de atualização: cliente 2 muda de cidade, cliente 6 é novo
schema_cli = StructType([
    StructField("id",     IntegerType(), False),
    StructField("nome",   StringType(),  True),
    StructField("email",  StringType(),  True),
    StructField("cidade", StringType(),  True),
    StructField("ativo",  BooleanType(), True),
])

novos_clientes = [
    (2, "Bruno Costa",  "bruno@email.com",  "Belo Horizonte", True),  # atualiza cidade
    (6, "Fernanda Paz", "fe@email.com",     "Recife",         True),  # novo cliente
]

df_merge = spark.createDataFrame(novos_clientes, schema_cli)
df_merge.createOrReplaceTempView("clientes_atualizados")

spark.sql("""
    MERGE INTO local.ecommerce.clientes AS destino
    USING clientes_atualizados AS origem
    ON destino.id = origem.id
    WHEN MATCHED THEN
        UPDATE SET destino.cidade = origem.cidade
    WHEN NOT MATCHED THEN
        INSERT (id, nome, email, cidade, ativo)
        VALUES (origem.id, origem.nome, origem.email, origem.cidade, origem.ativo)
""")

print("✅ MERGE INTO executado! Bruno teve cidade atualizada, Fernanda foi inserida.")
spark.sql("SELECT * FROM local.ecommerce.clientes ORDER BY id").show()

[Stage 58:=================================>                       (7 + 5) / 12]

✅ MERGE INTO executado! Bruno teve cidade atualizada, Fernanda foi inserida.
+---+-------------+---------------+--------------+-----+
| id|         nome|          email|        cidade|ativo|
+---+-------------+---------------+--------------+-----+
|  1|    Ana Silva|  ana@email.com|     São Paulo| true|
|  2|  Bruno Costa|bruno@email.com|Belo Horizonte| true|
|  3|  Carla Souza|carla@email.com|      Curitiba| true|
|  4|   Diego Lima|diego@email.com| Florianópolis|false|
|  5|Elena Martins|elena@email.com|  Porto Alegre| true|
|  6| Fernanda Paz|   fe@email.com|        Recife| true|
+---+-------------+---------------+--------------+-----+



## 9️⃣ Análise Final — Pedidos por Cliente

In [19]:
resultado = spark.sql("""
    SELECT
        c.nome,
        COUNT(p.id)   AS qtd_pedidos,
        SUM(p.total)  AS total_gasto
    FROM local.ecommerce.pedidos p
    JOIN local.ecommerce.clientes c ON p.cliente_id = c.id
    GROUP BY c.nome
    ORDER BY total_gasto DESC
""")

print("📊 Resumo de pedidos por cliente:")
resultado.show()

📊 Resumo de pedidos por cliente:
+-------------+-----------+-----------+
|         nome|qtd_pedidos|total_gasto|
+-------------+-----------+-----------+
|    Ana Silva|          2|     3679.8|
|  Carla Souza|          2|      600.0|
|Elena Martins|          1|      350.0|
|  Bruno Costa|          1|      269.7|
+-------------+-----------+-----------+



In [20]:
# Encerrar sessão Spark
spark.stop()
print("🔴 Sessão Spark encerrada.")

🔴 Sessão Spark encerrada.
